# Stage 4 — SFT v3 — Kaggle T4 x2 (Final)

## Fix lịch sử lỗi:
- ❌ lora_r=32 vs checkpoint r=16 → size mismatch → `lora_r=16`
- ❌ seq=2048+batch=2 → OOM training → `seq=1024, batch=1`
- ❌ eval logits convert fp32 → OOM evaluation → `eval_strategy="no"`

## Config an toàn:
| Param | Giá trị |
|---|---|
| seq_length | 1024 |
| lora_r | 16 (khớp checkpoint) |
| batch/GPU | 1 |
| eff. batch | 1 × 2 GPUs × 8 accum = **16** |
| eval | **OFF** (tránh OOM) |

In [1]:
# Cell 0 — Install
%pip install -q "unsloth[kaggle-new]" wandb huggingface_hub pyvi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 57.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 86.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.0 MB/s eta 0:00:00:00:01


In [2]:
# Cell 1 — Environment + Config
import os, sys, gc, json, math
import torch

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({vram:.1f} GB)")
else:
    raise RuntimeError("❌ No GPU!")

# ---- HF Token ----
HF_SECRET_NAMES = ["HF_TOKEN", "HF_TOKEN BEST", "HF_TOKEN_NEW"]
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for sname in HF_SECRET_NAMES:
        try:
            tok = sc.get_secret(sname)
            if tok:
                HF_TOKEN = tok
                print(f"HF_TOKEN via '{sname}': {tok[:8]}...")
                break
        except: continue
    if not HF_TOKEN:
        print("⚠️  No HF token found.")
except Exception as e:
    print(f"Secrets error: {e}")

# ---- Dataset paths ----
STAGE2_DATASET = "/kaggle/input/datasets/giahuyvotran/egal-llm-stage2-processed"
SFT_FILES = {
    "sft_train":     f"{STAGE2_DATASET}/data_processed/sft/sft_train.jsonl",
    "vn_legal_qa":   f"{STAGE2_DATASET}/data_processed/sft/sft_vn_legal_qa.jsonl",
    "vn_legal_chat": f"{STAGE2_DATASET}/data_processed/sft/sft_vn_legal_chat.jsonl",
}
OUTPUT_BASE = "/kaggle/working"
MODEL_DIR   = f"{OUTPUT_BASE}/sft_model"
ADAPTER_DIR = f"{OUTPUT_BASE}/sft_adapter"
LOG_DIR     = f"{OUTPUT_BASE}/logs"
for d in [MODEL_DIR, ADAPTER_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

SYSTEM_PROMPT = (
    "Bạn là trợ lý pháp luật Việt Nam chuyên nghiệp. "
    "Hãy trả lời câu hỏi dựa trên quy định pháp luật hiện hành. "
    "Luôn trích dẫn điều luật cụ thể khi có thể."
)

n_gpus = torch.cuda.device_count()
CONFIG = {
    "base_model_id":  "unsloth/Qwen2.5-7B-bnb-4bit",
    "max_seq_length": 1024,

    # lora_r=16 — MUST match HuggingFace checkpoint!
    "lora_r":         16,
    "lora_alpha":     32,
    "lora_dropout":   0.0,
    "lora_target_modules": ["q_proj","k_proj","v_proj","o_proj",
                             "gate_proj","up_proj","down_proj"],

    "max_train_records": 25_000,
    "min_chars":         50,

    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs":   2,
    "max_steps":          -1,
    "warmup_steps":       100,
    "learning_rate":      5e-5,
    "max_grad_norm":      0.3,
    "lr_scheduler_type":  "cosine",
    "weight_decay":       0.01,
    "seed":               42,
    "logging_steps":      10,
    "save_steps":         500,

    "output_dir":         MODEL_DIR,
    "hf_model_id":        "vtgh1602/legal-llm-sft-v3-qwen25-7b",
    "hf_checkpoint_repo": "vtgh1602/legal-llm-sft-v3-checkpoints",
    "push_to_hub":        True,
}

eff_batch = CONFIG["per_device_train_batch_size"] * n_gpus * CONFIG["gradient_accumulation_steps"]
print(f"\nConfig:")
print(f"  GPUs: {n_gpus} | seq: {CONFIG['max_seq_length']} | lora_r: {CONFIG['lora_r']}")
print(f"  eff.batch: {eff_batch} | eval: OFF (prevents OOM)")

for k, v in SFT_FILES.items():
    print(f"  {k:<15}: {'✅' if os.path.exists(v) else '❌ NOT FOUND'}")

Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch : 2.10.0+cu128
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)
HF_TOKEN via 'HF_TOKEN': hf_AHVtF...

Config:
  GPUs: 2 | seq: 1024 | lora_r: 16
  eff.batch: 16 | eval: OFF (prevents OOM)
  sft_train      : ✅
  vn_legal_qa    : ✅
  vn_legal_chat  : ✅


In [3]:
# Cell 2 — Load & Format SFT Data (train only, no eval split)
import json as _json, random
from datasets import Dataset

random.seed(CONFIG["seed"])

def to_chatml(instruction, inp, output):
    user_msg = (instruction + "\n\n" + inp).strip() if inp else instruction.strip()
    return (
        "<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n"
        "<|im_start|>user\n" + user_msg + "<|im_end|>\n"
        "<|im_start|>assistant\n" + output + "<|im_end|>"
    )

def load_jsonl(path, label):
    records = []
    if not os.path.exists(path):
        print(f"  [SKIP] {label}"); return records
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = _json.loads(line)
                ins = r.get("instruction", r.get("prompt", ""))
                inp = r.get("input",       r.get("context", ""))
                out = r.get("output",      r.get("response", r.get("answer", "")))
                if ins and out and len(out) >= CONFIG["min_chars"]:
                    records.append(to_chatml(ins, inp, out))
            except: continue
    print(f"  {label:<15}: {len(records):>6,} records")
    return records

print("Loading datasets...")
all_texts = []
for label, path in SFT_FILES.items():
    all_texts.extend(load_jsonl(path, label))

all_texts = list(set(all_texts))
random.shuffle(all_texts)
train_texts = all_texts[:CONFIG["max_train_records"]]

train_dataset = Dataset.from_dict({"text": train_texts})
print(f"  Train: {len(train_texts):,} (no eval split — eval disabled to avoid OOM)")

Loading datasets...
  sft_train      : 30,348 records
  vn_legal_qa    : 29,141 records
  vn_legal_chat  :  2,651 records
  Train: 25,000 (no eval split — eval disabled to avoid OOM)


In [4]:
# Cell 3 — Load Base Model
import gc, torch
gc.collect()
for i in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model_id"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
    dtype=None,
)
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {alloc:.2f}/{total:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
  GPU 0: 5.59/15.6 GB
  GPU 1: 0.04/15.6 GB


In [5]:
# Cell 4 — Apply LoRA
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
    use_rslora=True,
)
_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {_train/1e6:.1f}M / {_total/1e6:.1f}M ({100*_train/_total:.2f}%)")

Unsloth 2026.4.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable: 40.4M / 4393.3M (0.92%)


In [6]:
# Cell 5 — Auto-Resume from HuggingFace checkpoint
from huggingface_hub import HfApi, snapshot_download
import shutil

def get_latest_step(repo_id, token):
    try:
        path = HfApi().hf_hub_download(
            repo_id=repo_id, filename="latest_step.txt",
            repo_type="model", token=token)
        with open(path) as f: return int(f.read().strip())
    except: return None

def download_ckpt(repo_id, step, base, token):
    local = os.path.join(base, f"checkpoint-{step}")
    os.makedirs(local, exist_ok=True)
    print(f"  Downloading checkpoint-{step}...")
    snapshot_download(
        repo_id=repo_id, repo_type="model", local_dir=local,
        allow_patterns=[f"checkpoint-{step}/*"], token=token,
        ignore_patterns=["*.msgpack", "flax_model*"])
    nested = os.path.join(local, f"checkpoint-{step}")
    if os.path.isdir(nested):
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), os.path.join(local, item))
        os.rmdir(nested)
    total_mb = sum(os.path.getsize(os.path.join(local, f)) for f in os.listdir(local)) / 1024**2
    print(f"  Ready: {total_mb:.0f} MB")
    return local

resume_checkpoint = None
if HF_TOKEN:
    step = get_latest_step(CONFIG["hf_checkpoint_repo"], HF_TOKEN)
    if step:
        print(f"✅ Resuming from step {step}")
        resume_checkpoint = download_ckpt(
            CONFIG["hf_checkpoint_repo"], step, CONFIG["output_dir"], HF_TOKEN)
    else:
        print("No checkpoint → Training from scratch")
else:
    print("No HF_TOKEN → Training from scratch")
print(f"resume = {resume_checkpoint}")

latest_step.txt:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

✅ Resuming from step 5000


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

  Ready: 245 MB
resume = /kaggle/working/sft_model/checkpoint-5000


In [7]:
# Cell 6 — Train
import gc, datetime, os, math, torch
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer as Trainer, UnslothTrainingArguments as TrainConfig
from transformers import TrainerCallback
from huggingface_hub import HfApi

import torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

gc.collect()
for i in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()
model.train()
IS_BF16 = is_bfloat16_supported()
print(f"BF16: {IS_BF16} | GPUs: {torch.cuda.device_count()}")

class NaNStopCallback(TrainerCallback):
    def __init__(self, patience=3):
        self.nan_count = 0; self.patience = patience
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return control
        loss = logs.get("loss")
        if loss is not None:
            if math.isnan(loss) or math.isinf(loss):
                self.nan_count += 1
                print(f"\n⚠️  NaN step={state.global_step} ({self.nan_count}/{self.patience})")
                if self.nan_count >= self.patience:
                    print("🔴 ABORTED!"); control.should_training_stop = True
            else:
                self.nan_count = 0
                if state.global_step % 50 == 0:
                    print(f"  ✅ step {state.global_step}: loss={loss:.4f}")
        return control

class PushCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if not HF_TOKEN: return control
        if hasattr(state, 'is_world_process_zero') and not state.is_world_process_zero:
            return control
        step = state.global_step
        recent = [l for l in state.log_history if "loss" in l]
        if recent and (math.isnan(recent[-1]["loss"]) or math.isinf(recent[-1]["loss"])):
            return control
        try:
            model.push_to_hub(CONFIG["hf_model_id"], token=HF_TOKEN)
            tokenizer.push_to_hub(CONFIG["hf_model_id"], token=HF_TOKEN)
            print(f"[UP] Adapter → step {step}")
        except Exception as e:
            print(f"[WARN] push: {e}")
        ckpt = os.path.join(args.output_dir, f"checkpoint-{step}")
        if os.path.isdir(ckpt):
            try:
                api = HfApi()
                api.create_repo(CONFIG["hf_checkpoint_repo"], repo_type="model", exist_ok=True, token=HF_TOKEN)
                api.upload_folder(folder_path=ckpt, repo_id=CONFIG["hf_checkpoint_repo"],
                                  path_in_repo=f"checkpoint-{step}", repo_type="model", token=HF_TOKEN)
                api.upload_file(path_or_fileobj=str(step).encode(), path_in_repo="latest_step.txt",
                                repo_id=CONFIG["hf_checkpoint_repo"], repo_type="model", token=HF_TOKEN)
                print(f"[OK] checkpoint-{step} → Hub")
                try:
                    tree = [e.path for e in api.list_repo_tree(CONFIG["hf_checkpoint_repo"],
                                repo_type="model", token=HF_TOKEN)
                            if e.path.startswith("checkpoint-") and "/" not in e.path
                            and not e.path.endswith(f"checkpoint-{step}")]
                    for o in sorted(tree)[:-1]:
                        api.delete_folder(path_in_repo=o, repo_id=CONFIG["hf_checkpoint_repo"],
                                          repo_type="model", token=HF_TOKEN)
                except: pass
            except Exception as e:
                print(f"[WARN] ckpt: {e}")
        return control

training_args = TrainConfig(
    output_dir=CONFIG["output_dir"],
    run_name=f"sft-v3-t4x2-{datetime.datetime.now().strftime('%m%d-%H%M')}",
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    gradient_checkpointing=True,
    optim="adamw_8bit",
    learning_rate=CONFIG["learning_rate"],
    max_grad_norm=CONFIG["max_grad_norm"],
    weight_decay=CONFIG["weight_decay"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    warmup_steps=CONFIG["warmup_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    max_steps=CONFIG["max_steps"],
    bf16=IS_BF16, fp16=not IS_BF16,
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    # ✅ KEY FIX: eval_strategy="no" prevents OOM from logits->fp32 conversion
    eval_strategy="no",
    load_best_model_at_end=False,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    seed=CONFIG["seed"],
    dataloader_num_workers=0,
    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_dataset,
    # No eval_dataset — eval is disabled
    args=training_args,
    callbacks=[NaNStopCallback(patience=3), PushCallback()],
)

n_gpus = torch.cuda.device_count()
_eff = CONFIG["per_device_train_batch_size"] * n_gpus * CONFIG["gradient_accumulation_steps"]
_spe = len(train_dataset) // _eff
print("=" * 60); print("SFT v3 — T4 x2 (eval OFF)"); print("=" * 60)
print(f"  Resume   : {resume_checkpoint or 'scratch'}")
print(f"  GPUs     : {n_gpus}  |  eff.batch: {_eff}")
print(f"  Steps/ep : {_spe:,}  |  Total: {_spe * CONFIG['num_train_epochs']:,}")
print(f"  Eval     : DISABLED (prevents OOM)")
print("=" * 60)

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

final_loss = train_result.training_loss
is_nan = math.isnan(final_loss) if isinstance(final_loss, float) else False
print(f"\n{'='*60}")
print(f"DONE | Steps: {train_result.global_step} | Loss: {final_loss:.4f}" if not is_nan else "NaN ⚠️")
print(f"Time: {train_result.metrics.get('train_runtime',0)/60:.1f} min")

BF16: False | GPUs: 2


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/25000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFT v3 — T4 x2 (eval OFF)
  Resume   : /kaggle/working/sft_model/checkpoint-5000
  GPUs     : 2  |  eff.batch: 16
  Steps/ep : 1,562  |  Total: 3,124
  Eval     : DISABLED (prevents OOM)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,000 | Num Epochs = 2 | Total steps = 6,250
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


...

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


[UP] Adapter → step 6250


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[OK] checkpoint-6250 → Hub

DONE | Steps: 6250 | Loss: 0.0871
Time: 345.7 min


In [8]:
# Cell 7 — Save & Push Final
import math, os, json as _json
final_loss = train_result.training_loss
is_nan = math.isnan(final_loss) if isinstance(final_loss, float) else False

if is_nan:
    print("⛔ NaN — skip")
else:
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Saved → {ADAPTER_DIR}")
    rpt = {
        "stage": "04_sft_v3_t4x2", "base": CONFIG["base_model_id"],
        "lora_r": CONFIG["lora_r"], "loss": final_loss,
        "steps": train_result.global_step,
        "runtime_min": train_result.metrics.get("train_runtime", 0) / 60,
        "n_gpus": torch.cuda.device_count(),
    }
    with open(f"{LOG_DIR}/stage04_report.json", "w") as f:
        _json.dump(rpt, f, indent=2)
    if CONFIG["push_to_hub"] and HF_TOKEN:
        model.push_to_hub(CONFIG["hf_model_id"], token=HF_TOKEN)
        tokenizer.push_to_hub(CONFIG["hf_model_id"], token=HF_TOKEN)
        print(f"✅ → {CONFIG['hf_model_id']}")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_adapter/tokenizer_config.json.


Saved → /kaggle/working/sft_adapter


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/vtgh1602/legal-llm-sft-v3-qwen25-7b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp7yzrf7vm/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ → vtgh1602/legal-llm-sft-v3-qwen25-7b


In [9]:
# Cell 8 — Inference Test
import math
final_loss = train_result.training_loss
is_nan = math.isnan(final_loss) if isinstance(final_loss, float) else False

if is_nan:
    print("⛔ Skip — NaN")
else:
    from unsloth import FastLanguageModel
    import torch
    FastLanguageModel.for_inference(model)
    EOS_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

    def ask(q):
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        inp = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=CONFIG["max_seq_length"]).to("cuda")
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=512, do_sample=False,
                                repetition_penalty=1.1, eos_token_id=EOS_ID,
                                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inp["input_ids"].shape[1]:],
                                skip_special_tokens=True).split("<|im_end|>")[0].strip()

    print("=" * 60); print("INFERENCE TEST"); print("=" * 60)
    for q in [
        "Điều kiện kết hôn theo pháp luật Việt Nam?",
        "Thời hạn tạm giam trong vụ án hình sự?",
    ]:
        print(f"\n[Q] {q}")
        print(f"[A] {ask(q)[:400]}")
        print("-" * 60)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST

[Q] Điều kiện kết hôn theo pháp luật Việt Nam?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[A] Theo Khoản 1 Điều 3 của Luật Hôn nhân và gia đình, việc kết hôn phải đáp ứng các điều kiện sau: (1) Hai bên đều đủ tuổi kết hôn; (2) Hai bên tự nguyện kết hôn; (3) Hai bên không thuộc trường hợp cấm kết hôn quy định tại Điều 5 của Luật này.

Lưu ý: Nội dung trên chỉ mang tính chất tham khảo, không phải tư vấn pháp lý chính thức. Hãy tham khảo luật sư có chuyên môn để được tư vấn cụ thể.�
useRaluse
------------------------------------------------------------

[Q] Thời hạn tạm giam trong vụ án hình sự?
[A] Theo Khoản 1 Điều 23 của Luật Thi hành tạm giữ, tạm giam, thời hạn tạm giam đối với người chưa thành niên phạm tội được tính theo tháng và không quá một phần hai thời hạn tạm giam tương ứng quy định tại Bộ luật tố tụng hình sự. Tuy nhiên, thời hạn tạm giam này không vượt quá 04 tháng nếu họ bị bắt vì phạm tội rất nghiêm trọng hoặc phạm tội đặc biệt nghiêm trọng.

Lưu ý: Nội dung trên chỉ mang tính
------------------------------------------------------------
